# 🤖 Intelligent Single-Agent AI System
## Powered by LangGraph · Qwen3 14B (LM Studio) · Gemini 2.5 Flash

---

# SECTION 1 — Introduction

## What is an AI Agent?
An **AI Agent** is an autonomous system that perceives its environment, reasons about goals, and takes actions (via tools) to accomplish tasks — without requiring step-by-step human instruction. Unlike a simple chatbot, an agent can plan, decide, and iterate.

## Tool Calling
**Tool calling** (a.k.a. function calling) lets the LLM invoke external functions — like web search, calculators, or APIs — instead of relying solely on its parametric knowledge. The LLM outputs a structured JSON call; the runtime executes it and feeds results back.

## Agentic Workflow
An **agentic workflow** chains multiple LLM calls and tool executions together. The agent:
1. Receives a user query
2. Plans which tools to use
3. Executes tools (possibly in multiple rounds)
4. Synthesises results into a final answer

## Reasoning Loop (ReAct)
The **ReAct** (Reasoning + Acting) pattern interleaves *Thought → Action → Observation* steps in a loop until the agent decides it has enough information to answer.

```
Thought:  I need to find TCS stock prices.
Action:   yfinance_tool("TCS.NS", days=7)
Obs:      [prices ...]
Thought:  Now I'll compute the average.
Action:   calculate("average([...])")
Obs:      ₹3,842.14
Thought:  I have the answer. Summarise.
Final:    "The 7-day average price of TCS is ₹3,842.14"
```

## Function Calling
Modern LLMs support **structured function calling** — the model generates `{"name": "tool_name", "arguments": {...}}` JSON that is parsed and executed by the framework (here, LangGraph handles this transparently).

---
# SECTION 2 — Environment Setup

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    "langchain",
    "langchain-community",
    "langchain-openai",        # for LM Studio (OpenAI-compatible)
    "langgraph",
    "google-generativeai",     # Gemini
    "langchain-google-genai",  # LangChain wrapper for Gemini
    "duckduckgo-search",
    "yfinance",
    "pandas",
    "numpy",
    "matplotlib",
    "python-dotenv",
    "pydantic",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All packages installed successfully.")

---
# SECTION 3 — Architecture Diagram

```mermaid
flowchart TD
    A([👤 User Query]) --> B[🧠 Qwen3 14B Agent\nLM Studio]
    B --> C{🔀 Tool Selector}
    C --> D[🔍 Web Search\nDuckDuckGo]
    C --> E[📈 Stock Tool\nyFinance]
    C --> F[🧮 Calculator\nPython eval]
    D --> G[💾 LangGraph State\nMemory]
    E --> G
    F --> G
    G --> H[✨ Gemini 2.5 Flash\nSummarizer]
    H --> I([📋 Final Response])
```

**Flow:**
1. User sends a natural-language query.
2. **Qwen3 14B** (via LM Studio) acts as the reasoning brain.
3. The agent selects the appropriate tool(s).
4. Tool outputs are stored in **LangGraph State** (memory).
5. **Gemini 2.5 Flash** synthesises a clean final answer.

---
# SECTION 4 — Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import json
import math
import logging
import operator
import statistics
from typing import Annotated, Any, Sequence
from dataclasses import dataclass, field
from datetime import datetime, timedelta

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from pydantic import BaseModel, Field

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_core.messages import (
    AIMessage, HumanMessage, SystemMessage, ToolMessage, BaseMessage
)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI          # LM Studio is OpenAI-compatible
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun

# ── LangGraph ─────────────────────────────────────────────────────────────────
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# ── Logging setup ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("agent")

print("✅ Imports successful.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

# Gemini API key  (hard-coded as per project spec)
GEMINI_API_KEY = "AQ.Ab8RN6KPNlN7kLSUP456nVwwKsvtCgdzVsZZ1N8hKhrxo90lTg"
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

# LM Studio runs a local OpenAI-compatible server on port 1234 by default.
# Make sure LM Studio is running with Qwen3-14B loaded before executing cells below.
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
LM_STUDIO_MODEL    = "qwen3-14b"   # Match the model name shown in LM Studio

print(f"LM Studio URL : {LM_STUDIO_BASE_URL}")
print(f"Agent model   : {LM_STUDIO_MODEL}")
print(f"Summarizer    : gemini-2.5-flash")

---
# SECTION 4 — Tool Creation

Three tools are defined using LangChain's `@tool` decorator so that LangGraph can bind them to the agent.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOOL 1 — Web Search (DuckDuckGo)
# ─────────────────────────────────────────────────────────────────────────────

_ddg = DuckDuckGoSearchRun()

@tool
def search_web(query: str) -> str:
    """
    Search the web using DuckDuckGo and return the top results as text.
    Use this tool when you need current information, news, or facts
    that are not available in training data.

    Args:
        query: The search query string.

    Returns:
        A string containing the top search results.
    """
    logger.info(f"[WebSearch] query='{query}'")
    try:
        result = _ddg.run(query)
        if not result or result.strip() == "":
            return "No results found for the given query."
        logger.info(f"[WebSearch] returned {len(result)} chars")
        return result[:3000]   # cap at 3 000 chars to stay within context
    except Exception as e:
        logger.error(f"[WebSearch] error: {e}")
        return f"Search failed: {str(e)}"

print("✅ Tool 1 — search_web ready")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOOL 2 — Calculator
# ─────────────────────────────────────────────────────────────────────────────

def _safe_eval(expression: str) -> float:
    """
    Safely evaluate a mathematical expression.
    Supports: basic arithmetic, sum(), average(), mean(), max(), min(),
    percentage(value, pct), sqrt(), abs(), round().
    """
    # Whitelist of safe builtins
    safe_globals = {
        "__builtins__": {},
        "sum": sum,
        "max": max,
        "min": min,
        "abs": abs,
        "round": round,
        "sqrt": math.sqrt,
        "pi": math.pi,
        "e": math.e,
        "math": math,
        "statistics": statistics,
        "mean": statistics.mean,
        "median": statistics.median,
        "stdev": statistics.stdev,
    }

    # Convenience aliases
    def average(values):
        vals = list(values)
        return sum(vals) / len(vals) if vals else 0.0

    def percentage(value, pct):
        return (value * pct) / 100

    safe_globals["average"] = average
    safe_globals["percentage"] = percentage

    # Block anything dangerous
    forbidden = ["import", "exec", "eval", "open", "os", "sys", "__"]
    for token in forbidden:
        if token in expression:
            raise ValueError(f"Forbidden token in expression: '{token}'")

    return eval(expression, safe_globals, {})


@tool
def calculate(expression: str) -> str:
    """
    Evaluate a safe mathematical expression and return the result.
    Supports: arithmetic (+, -, *, /, **), sum([...]), average([...]),
    mean([...]), max([...]), min([...]), percentage(value, pct),
    sqrt(x), abs(x), round(x, n).

    Examples:
        calculate("average([100, 200, 150])")  -> 150.0
        calculate("percentage(2400, 15)")       -> 360.0
        calculate("sum([10, 20, 30])")           -> 60
        calculate("sqrt(144)")                   -> 12.0

    Args:
        expression: A mathematical expression string.

    Returns:
        The computed result as a string.
    """
    logger.info(f"[Calculator] expression='{expression}'")
    try:
        result = _safe_eval(expression.strip())
        formatted = f"{result:.4f}" if isinstance(result, float) else str(result)
        logger.info(f"[Calculator] result={formatted}")
        return f"Result: {formatted}"
    except Exception as e:
        logger.error(f"[Calculator] error: {e}")
        return f"Calculation error: {str(e)}"

# Quick sanity check
print(calculate.invoke({"expression": "average([100, 200, 300])"}))
print(calculate.invoke({"expression": "percentage(2400, 15)"}))
print("✅ Tool 2 — calculate ready")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOOL 3 — Stock Price Tool (yFinance)
# ─────────────────────────────────────────────────────────────────────────────

# Convenience map: friendly name → Yahoo Finance ticker
TICKER_MAP = {
    "tcs":      "TCS.NS",
    "infosys":  "INFY.NS",
    "wipro":    "WIPRO.NS",
    "reliance": "RELIANCE.NS",
    "hdfc":     "HDFCBANK.NS",
    "apple":    "AAPL",
    "google":   "GOOGL",
    "microsoft":"MSFT",
    "amazon":   "AMZN",
    "tesla":    "TSLA",
}


def _resolve_ticker(name: str) -> str:
    """Map a friendly company name to its Yahoo ticker."""
    key = name.lower().strip()
    return TICKER_MAP.get(key, name.upper())   # fall back to uppercase as-is


@tool
def get_stock_prices(company_name: str, days: int = 7) -> str:
    """
    Fetch the closing stock prices for a company over the last N trading days.
    Returns a JSON string with dates, closing prices, and basic statistics.

    Args:
        company_name: Company name or ticker symbol (e.g. "TCS", "Infosys", "AAPL").
        days: Number of calendar days to look back (default 7).

    Returns:
        JSON string with ticker, prices list, average, min, max.
    """
    ticker_symbol = _resolve_ticker(company_name)
    logger.info(f"[StockTool] {company_name} → {ticker_symbol}, last {days} days")

    end_date   = datetime.today()
    start_date = end_date - timedelta(days=days + 5)   # buffer for weekends/holidays

    try:
        ticker = yf.Ticker(ticker_symbol)
        hist   = ticker.history(start=start_date.strftime("%Y-%m-%d"),
                                end=end_date.strftime("%Y-%m-%d"))

        if hist.empty:
            return json.dumps({"error": f"No data found for ticker '{ticker_symbol}'."})

        # Take only the last `days` trading-day rows
        hist = hist.tail(days)
        prices = [round(float(p), 2) for p in hist["Close"].tolist()]
        dates  = [str(d.date()) for d in hist.index.tolist()]

        result = {
            "ticker":    ticker_symbol,
            "company":   company_name,
            "days":      len(prices),
            "dates":     dates,
            "prices":    prices,
            "average":   round(sum(prices) / len(prices), 2),
            "min":       min(prices),
            "max":       max(prices),
            "currency":  ticker.fast_info.get("currency", "INR"),
        }
        logger.info(f"[StockTool] avg={result['average']}")
        return json.dumps(result)

    except Exception as e:
        logger.error(f"[StockTool] error: {e}")
        return json.dumps({"error": str(e)})


# Quick test
raw = get_stock_prices.invoke({"company_name": "TCS", "days": 7})
data = json.loads(raw)
print(f"TCS avg price (last 7 days): {data.get('average', 'N/A')}")
print("✅ Tool 3 — get_stock_prices ready")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOOL 4 — Summarizer Tool (Gemini 2.5 Flash)
# ─────────────────────────────────────────────────────────────────────────────

_gemini = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
)


@tool
def summarize(text: str) -> str:
    """
    Use Gemini 2.5 Flash to summarise a block of text into a concise,
    human-friendly answer. Use this as the final step after all
    data has been collected and computed.

    Args:
        text: The raw data, numbers, or intermediate results to summarise.

    Returns:
        A clear, concise summary string.
    """
    logger.info(f"[Summarizer] input length={len(text)}")
    try:
        prompt = (
            "You are a concise financial and data analyst assistant. "
            "Summarise the following information into a clear, professional, "
            "human-friendly paragraph. Include key numbers and insights.\n\n"
            f"{text}"
        )
        response = _gemini.invoke([HumanMessage(content=prompt)])
        summary  = response.content.strip()
        logger.info("[Summarizer] done")
        return summary
    except Exception as e:
        logger.error(f"[Summarizer] error: {e}")
        return f"Summarization failed: {str(e)}"


print("✅ Tool 4 — summarize (Gemini 2.5 Flash) ready")

---
# SECTION 5 — Agent State & LLM Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AGENT STATE  (LangGraph typed state — serves as memory)
# ─────────────────────────────────────────────────────────────────────────────

from typing import TypedDict

class AgentState(TypedDict):
    """
    Typed state that flows through every node in the LangGraph graph.

    messages  : Full conversation history (user + AI + tool calls/results).
                `add_messages` is a reducer that appends new messages rather
                than overwriting, giving us automatic conversation memory.
    user_query: The original user query (preserved for the summariser).
    iteration : Safety counter — prevents infinite reasoning loops.
    """
    messages:   Annotated[list[BaseMessage], add_messages]
    user_query: str
    iteration:  int


# ─────────────────────────────────────────────────────────────────────────────
# LLM — Qwen3 14B via LM Studio (OpenAI-compatible endpoint)
# ─────────────────────────────────────────────────────────────────────────────

# All tools the agent can call
TOOLS = [search_web, calculate, get_stock_prices, summarize]

agent_llm = ChatOpenAI(
    model=LM_STUDIO_MODEL,
    base_url=LM_STUDIO_BASE_URL,
    api_key="lm-studio",     # LM Studio doesn't validate the key — any string works
    temperature=0.1,         # low temperature for reliable tool-call JSON
    max_tokens=2048,
).bind_tools(TOOLS)          # bind tools so the model knows their signatures

print("✅ Qwen3 14B agent LLM configured (requires LM Studio running locally)")

---
# SECTION 6 — Agent Workflow (LangGraph)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SYSTEM PROMPT  — instructs the agent on reasoning style & tool usage
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an intelligent AI assistant with access to the following tools:

1. search_web(query)          — search the internet for current information
2. get_stock_prices(company_name, days) — retrieve historical stock closing prices
3. calculate(expression)      — evaluate math expressions safely
4. summarize(text)            — produce a clean final summary using Gemini 2.5 Flash

Follow the ReAct reasoning pattern:
  Thought → Action (tool call) → Observation → Thought → ... → Final Answer

Guidelines:
- Always use get_stock_prices for stock-related queries (NOT web search).
- Use calculate after fetching numbers to perform arithmetic.
- Use summarize as the LAST step to compose the final answer.
- Keep intermediate thoughts concise.
- Never make up numbers — always fetch real data first.
- If a tool fails, try an alternative approach and explain what happened.
"""


# ─────────────────────────────────────────────────────────────────────────────
# NODE 1 — Agent (LLM reasoning + tool selection)
# ─────────────────────────────────────────────────────────────────────────────

MAX_ITERATIONS = 10   # safety guard against infinite loops

def agent_node(state: AgentState) -> dict:
    """
    The main reasoning node. Calls the LLM with the current message history
    and returns either a tool-call request or a final text answer.
    """
    logger.info(f"[AgentNode] iteration={state['iteration']}")

    # Prepend the system prompt on the first call
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(messages)

    response = agent_llm.invoke(messages)
    logger.info(f"[AgentNode] tool_calls={bool(response.tool_calls)}")

    return {
        "messages":   [response],
        "user_query": state["user_query"],
        "iteration":  state["iteration"] + 1,
    }


# ─────────────────────────────────────────────────────────────────────────────
# NODE 2 — Tool Execution  (LangGraph built-in ToolNode)
# ─────────────────────────────────────────────────────────────────────────────

tool_node = ToolNode(tools=TOOLS)


# ─────────────────────────────────────────────────────────────────────────────
# ROUTING LOGIC — should we call a tool or stop?
# ─────────────────────────────────────────────────────────────────────────────

def should_continue(state: AgentState) -> str:
    """
    Routing function called after the agent node.
    Returns:
      'tools'  — if the last AI message contains tool calls
      END      — if the agent produced a final text response, or max iterations hit
    """
    last_message = state["messages"][-1]

    # Safety: stop if too many iterations
    if state["iteration"] >= MAX_ITERATIONS:
        logger.warning("[Router] Max iterations reached — stopping.")
        return END

    # If the LLM requested tool calls, route to the tool node
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        tool_names = [tc["name"] for tc in last_message.tool_calls]
        logger.info(f"[Router] routing to tools: {tool_names}")
        return "tools"

    logger.info("[Router] no tool calls — END")
    return END


print("✅ Agent nodes and routing logic defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BUILD THE LANGGRAPH
# ─────────────────────────────────────────────────────────────────────────────

workflow = StateGraph(AgentState)

# Register nodes
workflow.add_node("agent",  agent_node)
workflow.add_node("tools",  tool_node)

# Set entry point
workflow.set_entry_point("agent")

# Conditional edge: after agent → either call tools or END
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END},
)

# After tool execution → always go back to agent for next reasoning step
workflow.add_edge("tools", "agent")

# Compile the graph
graph = workflow.compile()

print("✅ LangGraph compiled successfully")
print()
print("Graph structure:")
print("  START → agent → [tools → agent]* → END")

---
# SECTION 7 — Memory & Conversation History

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MEMORY — stored in LangGraph State (ConversationBuffer equivalent)
# The `add_messages` reducer in AgentState automatically accumulates:
#   • HumanMessage  (user queries)
#   • AIMessage     (agent thoughts + tool-call requests)
#   • ToolMessage   (tool results / observations)
# This gives us full conversation replay & multi-turn memory for free.
# ─────────────────────────────────────────────────────────────────────────────

# We maintain a simple session store for multi-turn use
session_history: list[BaseMessage] = []


def run_agent(query: str, verbose: bool = True) -> str:
    """
    Run the agent on a user query, with error handling, retry logic,
    and optional verbose logging of intermediate steps.

    Args:
        query  : Natural language question from the user.
        verbose: If True, print each step as the graph streams it.

    Returns:
        The final answer string.
    """
    global session_history

    # Validate input
    if not query or not query.strip():
        return "⚠️  Please provide a non-empty query."

    # Append this turn's user message to history (multi-turn memory)
    session_history.append(HumanMessage(content=query))

    # Initial state
    initial_state: AgentState = {
        "messages":   list(session_history),
        "user_query": query,
        "iteration":  0,
    }

    final_answer = "No answer generated."

    # ── Retry wrapper ───────────────────────────────────────────────────────
    max_retries = 2
    for attempt in range(1, max_retries + 1):
        try:
            if verbose:
                print(f"\n{'═'*60}")
                print(f"  Query: {query}")
                print(f"{'═'*60}")

            # Stream events from the graph
            for event in graph.stream(initial_state, stream_mode="values"):
                last_msg = event["messages"][-1]

                if verbose:
                    if isinstance(last_msg, AIMessage):
                        if last_msg.tool_calls:
                            for tc in last_msg.tool_calls:
                                print(f"\n  🔧 Tool Call → {tc['name']}")
                                args_str = json.dumps(tc['args'], indent=4)
                                print(f"     Args: {args_str}")
                        elif last_msg.content:
                            print(f"\n  🧠 Agent: {last_msg.content[:300]}")

                    elif isinstance(last_msg, ToolMessage):
                        print(f"\n  📥 Tool Result [{last_msg.name}]:")
                        print(f"     {str(last_msg.content)[:300]}")

            # Extract final answer from last AIMessage with text content
            all_messages = event["messages"]
            for msg in reversed(all_messages):
                if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
                    final_answer = msg.content
                    break

            # Update session history with all new messages
            session_history = list(all_messages)

            if verbose:
                print(f"\n{'─'*60}")
                print(f"  ✅ Final Answer:\n  {final_answer}")
                print(f"{'─'*60}")

            return final_answer

        except Exception as e:
            logger.error(f"[run_agent] attempt {attempt} failed: {e}")
            if attempt == max_retries:
                return f"❌ Agent failed after {max_retries} attempts: {str(e)}"
            print(f"  ⚠️  Attempt {attempt} failed ({e}) — retrying…")


def clear_memory():
    """Reset the conversation history for a fresh session."""
    global session_history
    session_history = []
    print("🧹 Memory cleared.")


print("✅ run_agent() helper ready")

---
# SECTION 8 — Stock Price Example

**Query:** *"What is the average stock price of TCS during the last 7 days?"*

**Expected Workflow:**
1. Agent identifies TCS → ticker `TCS.NS`
2. Calls `get_stock_prices("TCS", 7)` → fetches real prices from yFinance
3. Calls `calculate("average([...])")` → computes the average
4. Calls `summarize(...)` → Gemini produces the final answer

In [ ]:
# Demo: TCS 7-day average stock price
clear_memory()
answer = run_agent("What is the average stock price of TCS during the last 7 days?")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualise TCS price history
# ─────────────────────────────────────────────────────────────────────────────

raw = get_stock_prices.invoke({"company_name": "TCS", "days": 14})
stock_data = json.loads(raw)

if "error" not in stock_data:
    dates  = stock_data["dates"]
    prices = stock_data["prices"]
    avg    = stock_data["average"]
    curr   = stock_data.get("currency", "INR")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(dates, prices, marker="o", color="steelblue", linewidth=2, label="Close Price")
    ax.axhline(avg, color="tomato", linestyle="--", linewidth=1.5, label=f"Avg: {curr} {avg:,.2f}")
    ax.set_title(f"TCS — Last {len(prices)} Trading Days", fontsize=14, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel(f"Price ({curr})")
    ax.legend()
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig("tcs_price_chart.png", dpi=120)
    plt.show()
    print(f"\n📊 TCS Average (last {len(prices)} days): {curr} {avg:,.2f}")
else:
    print("Could not fetch TCS data:", stock_data["error"])

---
# SECTION 9 — Advanced Features

Error handling, retry logic, input validation, and logging are already baked into:
- Each tool function (`try/except` + `logger.error`)
- `run_agent()` (retry loop up to `max_retries=2`)
- `_safe_eval()` (whitelist + forbidden-token check)
- `should_continue()` (max-iteration guard)

Below we demonstrate the failure-recovery and logging behaviour explicitly.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ADVANCED FEATURE DEMONSTRATIONS
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 50)
print("1. Input Validation — empty query")
print("=" * 50)
print(run_agent("", verbose=False))

print()
print("=" * 50)
print("2. Calculator — forbidden token blocked")
print("=" * 50)
print(calculate.invoke({"expression": "__import__('os').system('ls')"}))

print()
print("=" * 50)
print("3. Stock Tool — unknown ticker graceful fallback")
print("=" * 50)
print(json.loads(get_stock_prices.invoke({"company_name": "FAKECORP999", "days": 7})))

print()
print("=" * 50)
print("4. Calculator — valid complex expression")
print("=" * 50)
print(calculate.invoke({"expression": "round(sqrt(2) * percentage(5000, 7.5), 2)"}))

---
# SECTION 10 — Evaluation: Four Test Queries

| # | Query | Tools Expected |
|---|-------|----------------|
| 1 | Average TCS stock price (last 7 days) | `get_stock_prices` → `calculate` → `summarize` |
| 2 | 15% of 2400 | `calculate` → `summarize` |
| 3 | Summarise latest AI news | `search_web` → `summarize` |
| 4 | Average Infosys closing price this month | `get_stock_prices` → `calculate` → `summarize` |

In [ ]:
# ── Query 1 ──────────────────────────────────────────────────────────────────
clear_memory()
q1 = run_agent("What is the average stock price of TCS during last 7 days?")

In [ ]:
# ── Query 2 ──────────────────────────────────────────────────────────────────
clear_memory()
q2 = run_agent("What is 15% of 2400?")

In [ ]:
# ── Query 3 ──────────────────────────────────────────────────────────────────
clear_memory()
q3 = run_agent("Summarize the latest AI news.")

In [ ]:
# ── Query 4 ──────────────────────────────────────────────────────────────────
clear_memory()
q4 = run_agent("What is the average closing price of Infosys this month?")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

results = [
    {"#": 1, "Query": "Average TCS stock price (last 7 days)",        "Answer": q1},
    {"#": 2, "Query": "15% of 2400",                                  "Answer": q2},
    {"#": 3, "Query": "Latest AI news summary",                       "Answer": q3},
    {"#": 4, "Query": "Average Infosys closing price this month",     "Answer": q4},
]

df_results = pd.DataFrame(results)
df_results["Answer"] = df_results["Answer"].str[:120] + "…"  # truncate for display

print("\n" + "=" * 80)
print(" EVALUATION RESULTS")
print("=" * 80)
for _, row in df_results.iterrows():
    print(f"\n[Q{row['#']}] {row['Query']}")
    print(f"  → {row['Answer']}")
print("\n" + "=" * 80)

---
# ✅ Summary

| Component | Technology |
|-----------|------------|
| **Agent Framework** | LangGraph (StateGraph with conditional edges) |
| **Agent Brain** | Qwen3 14B via LM Studio (OpenAI-compatible) |
| **Web Search** | DuckDuckGoSearchRun (LangChain Community) |
| **Stock Data** | yFinance |
| **Calculator** | Safe Python `eval` with whitelist |
| **Summarizer** | Gemini 2.5 Flash (Google Generative AI) |
| **Memory** | LangGraph `AgentState` + `add_messages` reducer |
| **Reasoning** | ReAct pattern (Thought → Action → Observation loop) |
| **Safety** | Max iterations, retry logic, input validation, token whitelist |

The system is **fully modular** — swap any component (e.g. replace Qwen3 with Llama 3, or add new tools) without changing the graph structure.